In [97]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [84]:
df = pd.read_csv('weather.csv')
df.head()

,Day,Hour,Temperature,Relative Humidity,Wind Speed,Wind Direction
0,2,0,28.94,77,9.69,221.99
1,2,1,28.79,79,13.04,219.40
2,2,2,28.59,79,7.99,215.84
3,2,3,28.47,79,5.48,203.20
4,2,4,28.40,78,5.24,195.95


In [86]:
df

,Day,Hour,Temperature,Relative Humidity,Wind Speed,Wind Direction
0,2,0,28.94,77,9.69,221.99
1,2,1,28.79,79,13.04,219.40
2,2,2,28.59,79,7.99,215.84
3,2,3,28.47,79,5.48,203.20
4,2,4,28.40,78,5.24,195.95
...,...,...,...,...,...,...
187,9,19,28.66,77,23.51,229.97
188,9,20,28.53,78,22.73,229.50
189,9,21,28.44,78,22.41,226.30
190,9,22,28.44,79,22.67,223.07


In [99]:
def get_state(row):
    if row['Relative Humidity'] >= 78:
        return "Rainy"
    elif row['Relative Humidity'] >= 69:
        return "Cloudy"
    else:
        return "Sunny"

df['State'] = df.apply(get_state, axis=1)

In [100]:
df['State'].value_counts()

State
Cloudy    90
Rainy     59
Sunny     43
Name: count, dtype: int64

In [101]:
states = df['State'].values

In [102]:
from collections import defaultdict

transition_counts = defaultdict(lambda: defaultdict(int))

for i in range(len(states) - 1):
    current = states[i]
    next_state = states[i + 1]
    transition_counts[current][next_state] += 1

In [103]:
transition_matrix = {}

for state in transition_counts:
    total = sum(transition_counts[state].values())
    transition_matrix[state] = {
        next_state: count / total
        for next_state, count in transition_counts[state].items()
    }

In [105]:
tm_df = pd.DataFrame(transition_matrix).T.fillna(0)
tm_df

,Rainy,Cloudy,Sunny
Cloudy,0.133333,0.755556,0.111111
Rainy,0.810345,0.189655,0.000000
Sunny,0.000000,0.232558,0.767442


In [106]:
transition_matrix = {
    'Cloudy': {'Rainy': 0.133333, 'Cloudy': 0.755556, 'Sunny': 0.111111},
    'Rainy': {'Rainy': 0.810345, 'Cloudy': 0.189655, 'Sunny': 0.0},
    'Sunny': {'Rainy': 0.0, 'Cloudy': 0.232558, 'Sunny': 0.767442}
}

In [107]:
import random

def simulate_weather(start_state, steps):
    current_state = start_state
    result = [current_state]

    for _ in range(steps):
        probs = transition_matrix[current_state]
        next_states = list(probs.keys())
        probabilities = list(probs.values())

        current_state = random.choices(next_states, probabilities)[0]
        result.append(current_state)

    return result

In [117]:
simulation = simulate_weather(start_state="Rainy", steps=15)
print(simulation)

['Rainy', 'Rainy', 'Rainy', 'Rainy', 'Rainy', 'Rainy', 'Cloudy', 'Cloudy', 'Cloudy', 'Sunny', 'Sunny', 'Cloudy', 'Cloudy', 'Cloudy', 'Cloudy', 'Sunny']


In [113]:
from collections import Counter

Counter(simulation)

Counter({'Cloudy': 16, 'Rainy': 4, 'Sunny': 1})